In [1]:
!pip install mysql-connector-python

   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.5 MB 1.7 MB/s eta 0:00:10
   - -------------------------------------- 0.8/16.5 MB 1.7 MB/s eta 0:00:10
   --- ------------------------------------ 1.3/16.5 MB 1.9 MB/s eta 0:00:08
   ---- ----------------------------------- 1.8/16.5 MB 2.1 MB/s eta 0:00:07
   ------ --------------------------------- 2.6/16.5 MB 2.3 MB/s eta 0:00:07
   ------- -------------------------------- 3.1/16.5 MB 2.4 MB/s eta 0:00:06
   --------- ------------------------------ 3.9/16.5 MB 2.6 MB/s eta 0:00:05
   ------------ --------------------------- 5.0/16.5 MB 2.8 MB/s eta 0:00:05
   ------------- -------------------------- 5.8/16.5 MB 3.0 MB/s eta 0:00:04
   ---------------- ----------------------- 6.8/16.5 MB 3.1 MB/s eta 0:00:04
   ------------------ --------------------- 7.6/16.5 MB 3.2 MB/s eta 0:00:03
   ----------

In [2]:
import pandas as pd
import numpy as np
import mysql.connector

In [3]:
def connect_to_db():
    return mysql.connector.connect(
        host="localhost",
        user="root",
        password="root123!@#",
        database="sql_project"
    )

In [5]:
connect_to_db().is_connected()

True

In [6]:
db=connect_to_db()
cursor=db.cursor(dictionary=True)

In [7]:
cursor

In [8]:
cursor.execute("select count(*) as total_suppliers  from suppliers")

In [9]:
row=cursor.fetchone()

In [10]:
row

{'total_suppliers': 50}

In [ ]:
## Now lets create function out of it 

def get_basic_info(cursor):
    """
    Retrieve summary inventory and supply chain metrics.

    Args:
        cursor (mysql.connector.cursor.MySQLCursorDict): Cursor object with dictionary=True.

    Returns:
        dict: Dictionary of metric labels and their values.
    """

    queries = {
        "Total Suppliers": "SELECT COUNT(*) AS count FROM suppliers",

        "Total Products": "SELECT COUNT(*) AS count FROM products",

        "Total Categories Dealing": "SELECT COUNT(DISTINCT category) AS count FROM products",

        "Total Sale Value (Last 3 Months)": """
            SELECT ROUND(SUM(ABS(se.change_quantity) * p.price), 2) AS total_sale
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Sale'
              AND se.entry_date >= (
                  SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Total Restock Value (Last 3 Months)": """
            SELECT ROUND(SUM(se.change_quantity * p.price), 2) AS total_restock
            FROM stock_entries se
            JOIN products p ON se.product_id = p.product_id
            WHERE se.change_type = 'Restock'
              AND se.entry_date >= (
                  SELECT DATE_SUB(MAX(entry_date), INTERVAL 3 MONTH) FROM stock_entries)
        """,

        "Below Reorder & No Pending Reorders": """
            SELECT COUNT(*) AS below_reorder
            FROM products p
            WHERE p.stock_quantity < p.reorder_level
              AND p.product_id NOT IN (
                  SELECT DISTINCT product_id FROM reorders WHERE status = 'Pending')
        """
    }

    results = {}
    for label, query in queries.items():
        cursor.execute(query)
        row = cursor.fetchone()
        # Since row is a dictionary, extract the single value by getting the first value in dict.values()
        results[label] = list(row.values())[0]

    return results

